# Kapitel 4: Die Maschine lernt

## Bisher...

Aus 3,6 Mio. echten Amazon-Bewertungen haben wir eine Stichprobe gezogen (Kap. 1),
das Klassenungleichgewicht durch Undersampling korrigiert (Kap. 2)
und die Texte in TF-IDF-Vektoren verwandelt (Kap. 3).

## Die entscheidende Frage

Reicht das aus, damit eine Maschine Emotionen erkennt?
Wir trainieren ein **Logistic-Regression-Modell** und messen,
wie gut es auf *ungesehenen* Testdaten abschneidet.

## 4.1 Daten laden

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Klassifikation") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df = spark.read.parquet("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/tfidf_features.parquet")
print(f"Daten geladen: {df.count():,} Zeilen")
df.printSchema()

## 4.2 Train/Test-Split (80/20)

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training:  {train_df.count():,} Zeilen")
print(f"Test:      {test_df.count():,} Zeilen")
print("\nLabel-Verteilung (Training):")
train_df.groupBy("label").count().orderBy("label").show()

## 4.3 Logistic Regression trainieren

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="tfidf_features", labelCol="label", maxIter=20, regParam=0.1)

print("Training gestartet...")
lr_model = lr.fit(train_df)
print("Training abgeschlossen!")
print(f"Accuracy (Training):  {lr_model.summary.accuracy:.4f}")
print(f"Area under ROC:       {lr_model.summary.areaUnderROC:.4f}")

## 4.4 Vorhersagen auf Testdaten

In [ ]:
predictions = lr_model.transform(test_df)
predictions.select("label", "prediction", "probability", "text").show(10, truncate=60)

## 4.5 Evaluation

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

binary_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = binary_eval.evaluate(predictions)

multi_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
accuracy  = multi_eval.evaluate(predictions, {multi_eval.metricName: "accuracy"})
precision = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedPrecision"})
recall    = multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedRecall"})
f1        = multi_eval.evaluate(predictions, {multi_eval.metricName: "f1"})

print("=" * 45)
print("  EVALUATIONS-ERGEBNISSE (Testdaten)")
print("=" * 45)
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Precision (gew.):   {precision:.4f}")
print(f"  Recall (gew.):      {recall:.4f}")
print(f"  F1-Score (gew.):    {f1:.4f}")
print(f"  AUC-ROC:            {auc:.4f}")
print("=" * 45)

## 4.6 Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Dark Theme
BG = '#0E1117'; CARD = '#1A1F2E'; TEXT = '#E2E8F0'; MUTED = '#64748B'
TEAL = '#14B8A6'; TEAL_D = '#0D9488'; RED = '#EF4444'; GREEN = '#10B981'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'text.color': TEXT, 'axes.labelcolor': TEXT,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': '#2D3548',
    'font.size': 11, 'axes.titlesize': 16, 'axes.titleweight': 'bold',
})

cm_data = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").toPandas()
cm = np.zeros((2, 2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row["label"]), int(row["prediction"])] = int(row["count"])

cmap = LinearSegmentedColormap.from_list('teal', [CARD, TEAL_D, TEAL], N=256)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap=cmap, aspect='auto')
labels = ["Negativ (0)", "Positiv (1)"]
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels, fontsize=13)
ax.set_yticklabels(labels, fontsize=13)
ax.set_xlabel("Vorhersage", fontsize=14, labelpad=10)
ax.set_ylabel("Tatsächlich", fontsize=14, labelpad=10)
ax.set_title("Confusion Matrix", fontsize=18, pad=15)

for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                fontsize=24, fontweight="bold", color='white')
        label_text = ['TN', 'FP', 'FN', 'TP'][i * 2 + j]
        label_color = TEAL if label_text in ['TN', 'TP'] else RED
        ax.text(j, i + 0.28, label_text, ha="center", va="center",
                fontsize=11, color=label_color, style='italic')

for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout()
plt.savefig("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/05_confusion_matrix.png",
            dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()

print(f"\nTrue Negative:  {cm[0,0]:,}")
print(f"False Positive: {cm[0,1]:,}")
print(f"False Negative: {cm[1,0]:,}")
print(f"True Positive:  {cm[1,1]:,}")

## 4.7 Beispiel-Vorhersagen

In [ ]:
from pyspark.sql.functions import col

print("=== KORREKTE Vorhersagen ===")
predictions.filter(col("label") == col("prediction")) \
    .select("label", "prediction", "text").show(5, truncate=80)

print("=== FALSCHE Vorhersagen ===")
predictions.filter(col("label") != col("prediction")) \
    .select("label", "prediction", "text").show(5, truncate=80)

## 4.8 Ergebnisse speichern

In [ ]:
output_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/predictions.parquet"
predictions.select("label", "prediction", "probability", "text") \
    .write.parquet(output_path, mode="overwrite")
print(f"Vorhersagen gespeichert: {output_path}")

model_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/lr_model"
lr_model.write().overwrite().save(model_path)
print(f"Modell gespeichert: {model_path}")

## Kapitel 4 — Zusammenfassung

**Die Antwort:** Ja, eine Maschine kann Emotionen erkennen — auch auf echten,
zuvor unbalancierten Trainingsdaten. Nach Bereinigung und Balancierung
erreicht Logistic Regression solide Ergebnisse.

**Nächstes Kapitel:** Wir visualisieren die gesamte Reise und ziehen ein Fazit.